In [1]:
import pandas as pd
import csv
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.stats import pearsonr, spearmanr
from disease import Disease
import networkx as nx

In [2]:
filename = 'data/final_enr.csv'
columns = ['diseaseId', '#snp', '#border_snp', 'pval_tad', 'pval_border', 'pval_outside', 'is_cancer', 'window_size', 'filter']
df = pd.read_csv(filename, usecols=columns)
print(df.head)

FileNotFoundError: [Errno 2] No such file or directory: 'data/final_enr.csv'

In [3]:
df = df[(df['#snp'] != 1) & (df['#border_snp'] != 1)]
df.shape

(52196, 9)

In [4]:
df = df[df['#border_snp'] != 0]
df.shape

(43950, 9)

In [5]:
df_cancers = pd.read_csv("data/cancerous_clustering.csv", usecols=['efoID', 'z_score', 'pvalue', 'p_empirical', 'name'])
df_cancers = df_cancers.rename(columns={'efoID': 'diseaseId'})
df_cancers.shape

(32, 5)

In [6]:
df_noncancers = pd.read_csv("data/noncancerous_clustering.csv", usecols=['efoID', 'z_score', 'pvalue', 'p_empirical', 'name'])
df_noncancers = df_noncancers.rename(columns={'efoID': 'diseaseId'})
df_noncancers.shape

(247, 5)

In [7]:
combined = pd.concat([df_cancers, df_noncancers])

In [8]:
combined.head()

,name,diseaseId,z_score,pvalue,p_empirical
0,basal cell carcinoma,EFO_0004193,4.690213,0.000031,0.001998
1,squamous cell carcinoma,EFO_0000707,3.968751,0.001300,0.009990
2,keratinocyte carcinoma,EFO_0010176,5.038859,0.004000,0.001998
3,cutaneous melanoma,EFO_0000389,15.667010,0.000000,0.000999
4,colorectal cancer,MONDO_0005575,0.216129,0.002900,0.401598


In [9]:
combined.shape

(279, 5)

In [10]:
def map_genes(row):
    try:
        d = Disease(row['name'], row['diseaseId'])
        genes = d.get_genes()
        return ",".join(genes) if genes else ""
    except Exception as e:
        print(f"Problem with {row['diseaseId']}: {e}")
        return ""


combined['mapped-genes'] = combined.apply(map_genes, axis=1)
combined.to_csv('data/combined_diseases_genes.csv', sep='\t', index=False)

KeyboardInterrupt: 

In [3]:
mapped_genes = pd.read_csv('data/combined_diseases_genes.csv', delimiter='\t')

In [4]:
mapped_genes.head()

,name,diseaseId,z_score,pvalue,p_empirical,mapped-genes
0,basal cell carcinoma,EFO_0004193,4.690213,0.000031,0.001998,"TMCC3,KRT19P2,GRXCR1,LINC02383,CSMD1,DTNBP1,AR..."
1,squamous cell carcinoma,EFO_0000707,3.968751,0.001300,0.009990,"SLC45A2,ANKRD11,RPS18P8,RPP40,MICA,LINC01149,L..."
2,keratinocyte carcinoma,EFO_0010176,5.038859,0.004000,0.001998,"RAET1L,RAET1M,PIK3R1,LINC02198,TRPS1,DHX35,LIN..."
3,cutaneous melanoma,EFO_0000389,15.667010,0.000000,0.000999,"RNA5SP293,RPS15AP27,TYR,GPRC5A,MTAP,FMN1,FTO,R..."
4,colorectal cancer,MONDO_0005575,0.216129,0.002900,0.401598,"TERT,MIR4457,RNU1,150P,TTC33,LAMC1,LINC01705,T..."


In [5]:
import networkx as nx
import pandas as pd
from tqdm import tqdm

df = pd.read_csv("data/whole_network.txt", delim_whitespace=True)

G = nx.Graph()
for idx, row in tqdm(df.iterrows(), total=len(df)):
    G.add_edge(row["protein1"], row["protein2"], combined_score=row["combined_score"])

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

for u, v, d in tqdm(G.edges(data=True), total=G.number_of_edges()):
    d["inv_weight"] = 1 / d["combined_score"]

centrality = nx.betweenness_centrality(G, weight="inv_weight", k=500, seed=42)

print("Done!")
print(centrality)


/var/folders/4g/n465f8j14bq_nt2x3wjkt8pw0000gn/T/ipykernel_76551/1189350060.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv("data/whole_network.txt", delim_whitespace=True)
100%|██████████| 13715404/13715404 [06:28<00:00, 35273.89it/s]


Nodes: 19622, Edges: 6857702


100%|██████████| 6857702/6857702 [00:04<00:00, 1558475.79it/s]


KeyboardInterrupt: 

In [ ]:
print(G.number_of_nodes())
print(G.number_of_edges())


0
0


In [ ]:
aliases_df = pd.read_csv('data/9606.protein.aliases.v12.0.txt', sep='\t')

In [6]:
aliases_df

,string_protein_id,alias,source
0,9606.ENSP00000000233,2B6H,Ensembl_PDB
1,9606.ENSP00000000233,2B6H,UniProt_DR_PDB
2,9606.ENSP00000000233,381,Ensembl_HGNC_entrez_id
3,9606.ENSP00000000233,381,KEGG_GENEID
4,9606.ENSP00000000233,381,KEGG_KEGGID_SHORT
...,...,...,...
3889202,9606.ENSP00000501317,regulatory factor X domain containing 2,Ensembl_HGNC_prev_name
3889203,9606.ENSP00000501317,"regulatory factor X, 7",Ensembl_HGNC_prev_name
3889204,9606.ENSP00000501317,regulatory factor X7,Ensembl_HGNC_name
3889205,9606.ENSP00000501317,uc059jng.1,Ensembl_HGNC_ucsc_id


In [7]:
centrality

{'9606.ENSP00000000233': 2.0796166872945687e-05,
 '9606.ENSP00000356607': 2.4466078674053748e-05,
 '9606.ENSP00000427567': 9.174779502770155e-07,
 '9606.ENSP00000253413': 5.1276823221037645e-05,
 '9606.ENSP00000493357': 8.787399923764305e-05,
 '9606.ENSP00000324127': 2.2427238784549267e-06,
 '9606.ENSP00000325266': 1.6922371082887175e-05,
 '9606.ENSP00000320935': 0.0003945155186191167,
 '9606.ENSP00000371175': 7.360212001111169e-05,
 '9606.ENSP00000480364': 6.25923846077875e-05,
 '9606.ENSP00000388107': 0.006837657337431171,
 '9606.ENSP00000461784': 8.98108971326723e-05,
 '9606.ENSP00000381097': 2.1305876845321805e-05,
 '9606.ENSP00000234160': 3.996126183428779e-05,
 '9606.ENSP00000241502': 9.684489475146274e-06,
 '9606.ENSP00000317272': 0.0003005249997129602,
 '9606.ENSP00000487719': 5.097099723761198e-06,
 '9606.ENSP00000354878': 1.1009735403324187e-05,
 '9606.ENSP00000479606': 0.0010947041076721923,
 '9606.ENSP00000369127': 4.424282560224719e-05,
 '9606.ENSP00000493946': 0.000347520

In [9]:
alias_to_protein = dict(zip(aliases_df['alias'], aliases_df['string_protein_id']))

In [10]:
expanded = mapped_genes.copy()
expanded["mapped-genes"] = expanded["mapped-genes"].str.split(",")
expanded = expanded.explode("mapped-genes")

expanded["protein_id"] = expanded["mapped-genes"].map(alias_to_protein)

expanded["centrality"] = expanded["protein_id"].map(centrality).fillna(0)

mean_centrality = expanded.groupby(expanded.index)["centrality"].mean()

mapped_genes["mean_centrality"] = mean_centrality

In [11]:
mapped_genes

,name,diseaseId,z_score,pvalue,p_empirical,mapped-genes,mean_centrality
0,basal cell carcinoma,EFO_0004193,4.690213,3.120000e-05,0.001998,"TMCC3,KRT19P2,GRXCR1,LINC02383,CSMD1,DTNBP1,AR...",0.000209
1,squamous cell carcinoma,EFO_0000707,3.968751,1.300000e-03,0.009990,"SLC45A2,ANKRD11,RPS18P8,RPP40,MICA,LINC01149,L...",0.000228
2,keratinocyte carcinoma,EFO_0010176,5.038859,4.000000e-03,0.001998,"RAET1L,RAET1M,PIK3R1,LINC02198,TRPS1,DHX35,LIN...",0.000423
3,cutaneous melanoma,EFO_0000389,15.667010,0.000000e+00,0.000999,"RNA5SP293,RPS15AP27,TYR,GPRC5A,MTAP,FMN1,FTO,R...",0.000296
4,colorectal cancer,MONDO_0005575,0.216129,2.900000e-03,0.401598,"TERT,MIR4457,RNU1,150P,TTC33,LAMC1,LINC01705,T...",0.000068
...,...,...,...,...,...,...,...
274,erythematosquamous dermatosis,EFO_1000695,14.059322,2.390000e-11,0.000999,"CARD14,TYR,IL12B,LINC01845,IRF4,ZMIZ1,IL2RA,IL...",0.000051
275,color vision disorder,MONDO_0001703,0.298102,6.790000e-07,0.350649,"AMZ1,NRF1,SFRP4,STARD3NL,ZNF775,RABGEF1,SEMA3D...",0.000056
276,QRS-T angle,EFO_0020097,4.241194,1.200000e-03,0.007992,"HAND1,CIR1P1,HNRNPLP1,RPS26P29,PPP1R3B,DT,KLHL...",0.000083
277,pregnancy disorder,EFO_0009682,1.421844,6.900000e-03,0.117882,"HELZ2,GMEB2,ZSCAN2,AS1,MAST4,TRMU,SLC1A3,PURPL...",0.000055


In [13]:
mapped_genes.to_csv("data/mean_centrality_test.csv", sep='\t')